# Proyecto TELECOM

## Informe de solución

### Descipción ejecutiva

**Problema del negocio abordado.**

En la industria de las telecomunicaciones, la tasa de cancelación de clientes (Customer Churn) representa uno of los desafíos operativos y financieros más críticos. Para la empresa, el 26.5% de la base de clientes actual abandona el servicio, lo que genera una pérdida directa de ingresos recurrentes y afecta la sostenibilidad del modelo de negocio (Jain & Surana, 2017).

El problema central no radica únicamente en la pérdida de clientes, sino en la incapacidad de reaccionar preventivamente antes de que la cancelación ocurra. Hasta ahora, la compañía ha operado de forma reactiva, intentando recuperar usuarios una vez que el contrato ha finalizado o el servicio ha sido cancelado, momento en el cual el costo de retención es sustancialmente más elevado y la efectividad es mínima.


**Importancia de su resolución y análisis realizado.**

La resolución de este problema mediante modelos analíticos y analítica predictiva aporta valor estratégico en tres frentes fundamentales:

* Eficiencia en la adquisición vs. retención: Financieramente, captar un cliente nuevo es entre 5 y 25 veces más costoso que fidelizar a un cliente existente. Reducir la tasa de Churn optimiza de inmediato el Lifetime Value (LTV) del usuario y maximiza el retorno de inversión en mercadeo y ventas (Gallo, 2014).

* Capacidad de acción proactiva: Mediante el desarrollo del modelo predictivo LightGBM, la empresa ahora es capaz de identificar anticipadamente a 4 de cada 5 clientes en riesgo real de abandono (Recall = 80.48%), permitiendo al equipo de retención intervenir de forma oportuna con ofertas focalizadas.

* Decisiones basadas en evidencia (análisis SHAP): El análisis interpretativo reveló los disparadores exactos del Churn en la compañía:

* Vulnerabilidad temprana: La baja antigüedad (TenureDays) es el mayor factor de riesgo.

* Inestabilidad contractual: Los contratos mes a mes (Month-to-month) concentran las fugas frente a la estabilidad que brindan los contratos de 1 y 2 años.

* Fricción en servicios de alto valor: La tarifa mensual (MonthlyCharges) y los planes de Fibra Óptica registran un riesgo proporcionalmente alto de cancelación, señalando posibles problemas de competitividad en precio o percepción de calidad.

Resolver este problema transforma la gestión de clientes: pasa de ser un proceso defensivo e incierto a una estrategia predictiva, rentable y basada en datos.

* Referencias:
    * Gallo, A. (2014). The value of keeping the right customers. Harvard business review, 29(10), 304-309.
    * Jain, P., & Surana, K. (2017). Reducing churn in telecom through advanced analytics. McKinsey & Company.  

#### Descripción ejecutiva de los datos utilizados

**Nivel de calidad general encontrado**

El conjunto de datos procesado (TELECOM Dataset) consta originalmente de 7,043 registros y 21 atributos, representando la actividad contractual, demográfica y financiera de los clientes. Tras la fase de auditoría e ingeniería de datos, el nivel de calidad se evalúa como Alto y Estructuralmente Sólido, destacando los siguientes aspectos:

* Integridad de registros: No se registraron filas duplicadas ni ausencias masivas de información. Las variables categóricas de servicios adicionales (InternetService, MultipleLines, TechSupport, etc.) mantenían una coherencia lógica perfecta respecto a los servicios contratados.

* Corregibilidad de anomalías: Los valores nulos e inconsistencias detectadas en la columna TotalCharges (asociados a espacios en blanco " " en clientes con antigüedad cero) se resolvieron mediante imputación lógica basada en la tarifa mensual (MonthlyCharges), alcanzando un dataset final con 0% de valores faltantes (7,043/7,043 registros válidos).

* Consistencia de variables calculadas: Se logró transformar exitosamente las variables temporales de ingreso (BeginDate) y salida (EndDate) en una métrica cuantitativa continua pura (TenureDays), eliminando el riesgo de fuga de datos (data leakage) durante el modelado.


**Limitaciones derivadas de los datos**

A pesar de la alta calidad técnica alcanzada, el conjunto de datos presenta limitaciones de negocio y de estructura que deben ser tomadas en cuenta al interpretar las predicciones:

* Desbalanceo de clases moderado (~26.5% de Churn):

    * Limitación: La clase objetivo presenta una distribución asimétrica (aproximadamente 1 cliente que abandona por cada 3 que permanecen).

    * Impacto: Aunque no es un desbalanceo extremo, requirió el uso de técnicas de ajuste de pesos (scale_pos_weight / class_weight='balanced') para evitar que los algoritmos favorecieran sistemáticamente la predicción de la clase mayoritaria (Permanecer).

* Naturaleza estática del dataset (Corte Transversal):

    * Limitación: Los datos representan una "fotografía" en un punto fijo del tiempo y no incluyen métricas de uso dinámico en tiempo real (por ejemplo: consumo de datos del último mes, número de llamadas a atención al cliente en los últimos 30 días o caídas registradas en el servicio).

    * Impacto: El modelo predice la propensión al Churn basándose en la configuración del contrato y facturación, pero no puede capturar cambios abruptos en el comportamiento reciente del cliente.

* Inexistencia de indicadores de calidad de servicio / satisfacción:

    * Limitación: El hallazgo de que el servicio de Fibra Óptica incrementa el riesgo de Churn sugiere un problema de satisfacción o costo, pero el dataset no cuenta con columnas de quejas, tickets de soporte resueltos o encuestas de satisfacción.

    * Impacto: El modelo identifica que la Fibra Óptica es un factor de riesgo, pero no puede diagnosticar la causa raíz exacta (si es por mal servicio técnico o por precio elevado).

#### Descripción técnica de la solución planteada

**Métricas clave y otros indicadores utilizados**

Para evaluar el rendimiento técnico y asegurar que el modelo responda a las necesidades de la estrategia de retención, se utilizaron las siguientes métricas:

* ROC-AUC (Área Bajo la Curva ROC - Métrica Principal de Selección): Evalúa la capacidad global del modelo para discriminar entre clientes que cancelarán ($Churn = 1$) y aquellos que permanecerán ($Churn = 0$) a lo largo de distintos umbrales de decisión.
* Recall / Sensibilidad (Métrica Clave de Negocio): Mide la proporción de clientes en riesgo real de Churn que el modelo identifica correctamente. Es la métrica prioritaria para minimizar los falsos negativos (clientes que se van sin ser detectados).
* Precision: Mide el porcentaje de clientes verdaderamente en riesgo entre todos los que el modelo etiquetó como propensos al Churn, permitiendo controlar los costos de falsos positivos en campañas de retención.
* F1-Score: Representa la media armónica entre Precision y Recall, garantizando un equilibrio operativo sostenido.Accuracy: Utilizada como indicador secundario de alineación global del modelo.

**Mejor modelo aplicado**

Algoritmo Seleccionado: **LightGBM** (LGBMClassifier), un algoritmo basado en Gradient Boosted Decision Trees que utiliza un crecimiento de árboles enfocado en hojas (leaf-wise), ofreciendo una alta velocidad de cómputo y una capacidad superior para capturar relaciones no lineales complejas.

* Tratamiento del Desbalanceo: Se aplicó un balanceo nativo mediante el hiperparámetro scale_pos_weight = ratio_desbalanceo ($\approx 2.77$), ajustado automáticamente según la proporción entre la clase negativa y la clase positiva en el conjunto de entrenamiento. Esto evita la necesidad de inventar datos sintéticos (como SMOTE) y previene la fuga de información.
* Calibración e Hiperparámetros Óptimos (mediante GridSearchCV con 5-Fold CV):
    * n_estimators: 100 (número óptimo de árboles de decisión).
    * learning_rate: 0.1 (tasa de aprendizaje para la convergencia).
    * num_leaves: 31 (complejidad máxima de cada árbol).
    * random_state: 42 (garantía de reproducibilidad).

**Rendimiento técnico y comparación con otros modelos**

El proceso experimental se diseñó dividiendo los datos en un conjunto de Entrenamiento (60%), Prueba (20%) y Validación (20%), aplicando escalado (StandardScaler) únicamente sobre las variables continuas (MonthlyCharges, TotalCharges, TenureDays) ajustado sobre Train para evitar Data Leakage.

**Verificación final con el conjunto de validación**

Tras re-entrenar el modelo LightGBM con los hiperparámetros óptimos combinando el conjunto de Train + Test, los resultados finales obtenidos sobre el conjunto de Validación completamente independiente confirmaron la solidez de la solución:

LightGBM demostró un dominio absoluto frente a los modelos baseline y competidores tradicionales, alcanzando un ROC-AUC de 0.9114 y un Recall del 80.48%, demostrando una capacidad sobresaliente de generalización sin sobreajuste.

A continuación se observan los resultados de los diversos modelos entrenados, incluido la versión final del modelo ganador entrenado con el conjunto de entrenamiento y prueba y evaluado con el conjunto de validación, también se muestra la matriz de confusión y la curva ROC del modelo final:

![Mi imagen](grafica_metricas_finales_completa.png)

![Mi imagen](resultados_validacion.png)

Se muestran también las características más informativas para la predicción del modelo final (primero las obtenidas por el mismo modelo y después las obtenidas por la herramienta SHAP), las cuales serán de relevancia más adelante para las recomendaciones al negocio.

![Mi imagen](top_features.png)
![Mi imagen](shap.png)

#### Estimación del Impacto del Modelo en el Negocio

Para traducir el rendimiento técnico de LightGBM ($Recall = 80.48\%$, $Precision = 65.43\%$) a un valor económico tangible, se realiza una estimación cuantitativa basada en una muestra representativa equivalente al conjunto de validación ($1,409$ clientes): 

* **Asunciones del escenario financiero (Base de Validación: 1,409 clientes)**
    * Tasa real de Churn en la muestra: $26.5\%$ ($\approx 373$ clientes que realmente abandonan).
    * Valor promedio mensual por cliente ($ARPU$), obtenido a tráves de la media de MonthlyCharges: $\$65.00$ USD/mes ($\approx \$780.00$ USD/año por cliente).
    * Efectividad de la campaña de retención (Incentivo/Descuento): Supongamos que una oferta atractiva logra retener al $50\%$ de los clientes contactados que realmente se iban a ir.
    * Costo del incentivo de retención: $\$100.00$ USD por cliente contactado (aplicado a verdaderos y falsos positivos).
    
* **Cálculo del impacto financiero anualizado**

* Retención de Clientes en Riesgo Real (Verdaderos Positivos):

    * El modelo detecta al $80.48\%$ de los $373$ clientes en riesgo (debido a su valor de 0.8 en Recall, que se interpreta como que se detectarán 8 de cada 10 clientes que cancelarán) $\rightarrow$ $300$ clientes identificados.
    * Si la campaña retiene al $50\%$, salvamos a $150$ clientes.
    * Ingresos Salvados: $150 \text{ clientes} \times \$780 \text{ USD/año} =$ $+\$117,000\text{ USD/año}$.
      

* **Costo Operativo de la Campaña de Retención**

    * El modelo genera $300$ Verdaderos Positivos + $158$ Falsos Positivos $\rightarrow$ $458$ ofertas emitidas.
    * Costo de Incentivos: $458 \text{ clientes} \times \$100 \text{ USD} =$ $-\$45,800\text{ USD}$.
* Beneficio Neto Estimado (ROI del Modelo):

    * $$\text{Beneficio Neto Anual} = \$117,000 - \$45,800 = \mathbf{+\$71,200\text{ USD}}$$
    
* Impacto Escala Negocio (Base Total de 7,000+ Clientes): Escalar este modelo a toda la base de clientes de la compañía generaría un beneficio neto estimado superior a los $\$350,000\text{ USD anuales}$, reduciendo la pérdida neta de ingresos por Churn en más de un $40\%$.


#### Principales conclusiones y recomendaciones para la toma de decisiones

* **Conclusiones Clave**
    * Desempeño del modelo LightGBM: El modelo superó de forma contundente al modelo base que predecía solamente la clase mayoritaria y a algoritmos tradicionales, alcanzando un ROC-AUC de 0.9114. La calibración mediante peso de clases (scale_pos_weight) eliminó la necesidad sub o sobre muestrear los datos, garantizando predicciones realistas sin un sesgo por el desbalance de clases.

    * El "Primer Año" es el punto crítico: El análisis SHAP demostró que la baja antigüedad (TenureDays) es la variable que más acelera el Churn, siendo más bajos los días de contrato entre los clientes que cancelan. Una vez que el cliente supera la barrera de los 12-24 meses, la probabilidad de fuga cae drásticamente.
 
    * El uso de métodos de pago no automatizados: Los métodos no automaticos como los cheques también resultaron ser más comunes entre los clientes que cancelan, esto se observó en la distribución de los datos en el EDA y se confirmó su relevancia apareciendo como una de las características más informativas para el mejor modelo (mediante su propia evaluación y SHAP). La relevancia de esto puede ser que sea fuente de cancelaciones involuntarias debido a falta de pago por situaciones como por ejemplo, olvidar la fecha de pago, algo que no ocurriría si su pago se encontrara domiciliado. Por esto se recomienda que entre las campañas para nuevos clientes y las que buscan atacar al sector de clientes que tienen mayor probabilidad de cancelar se incluyan también incentivos para que el usuario vincule su tarjeta para el pago automático.

    * Riesgo por contrato y producto: Los contratos Mensuales y los planes de Fibra Óptica combinados con tarifas elevadas concentran la mayor propensión a la cancelación. En el primer caso se podría incentivar tanto a usuarios nuevos como aquellos con alto riesgo de cancelar a que opten por un contrato más extenso, se puede hacer uso de tarifas especiales o descuentos que permitan un balance que permita mantener ganancias respecto a simplemente dejar ir al cliente. En el segundo caso se debe observar más detalladamente el contexto de esta tendencia de cancelación, ya que puede ser que por si mismo el servicio de fibra óptica sea más caro lo cual derivaría en pagos mensuales elevados (lo cual ya se observó como una de las condiciones vinculadas a cancelar) o que este servicio no ofrezca la calidad esperada por los clientes, por lo cual opten por cancelar. El contexto de relevancia de esta característica lamentablemente no se puede verificar con los datos disponibles actuales, por lo cual es una de las limitaciones del presente análisis.
 

* **Recomendaciones estratégicas y operativas**

    * Diseñar un programa de seguimiento y vigilancia para usuarios nuevos (días 1 a 180):
        * Acción: Concentrar los esfuerzos de servicio al cliente y seguimiento de satisfacción en los primeros 6 meses de contrato. Reducir la fricción inicial es la palanca de mayor impacto para extender el TenureDays y con lo cual reducir la probabilidades de deserción.
    * Estrategia de migración contractual:
        * Acción: Crear incentivos económicos (ej. un mes gratis o mejoras de velocidad) para clientes en modalidad de contrato mensual que acepten migrar a contratos de 1 o 2 años, bloqueando la fuga según lo demostrado por el análisis SHAP. Lo mismo puede ser aplicado para el registro de métodos de pago automáticos y evitar cancelaciones involuntarias.
    * Auditoría de calidad y precio en fibra óptica:
        * Acción: Investigar la causa raíz de la cancelación en clientes de Fibra Óptica. Si la razón es precio, ofrecer paquetes ligeramente más económicos o con otras ventajas que incentiven a su contratación; si la razón es técnica, priorizar la atención de tickets de soporte para esta tecnología.
    * Despliegue del sistema de alertas proactivas:
        * Acción: Integrar la salida probabilística del modelo LightGBM al CRM comercial. Todo cliente que alcance una probabilidad de Churn $\ge 50\%$ debe ser asignado automáticamente a la lista de llamadas del equipo de retención y ofrecer incentivos para su conservación.

#### ¿Qué pasos del plan se realizaron y qué pasos se omitieron (explica por qué)?

Pasos realizados:

* Limpieza e ingeniería de datos: Conversión de tipos de datos (TotalCharges), imputación lógica basada en el cobro mensual (MonthlyCharges), creación de la métrica TenureDays y eliminación de variables de fecha/IDs sin valor predictivo.

* Codificación y escalado: One-Hot Encoding (pd.get_dummies) en categóricas y escalado estándar (StandardScaler) en numéricas.

* Partición estratificada: División 60% Train, 20% Test y 20% Validation para evitar fugas de información y conservando las proporciones de clases del conjunto original.

* Modelo base para comparación y optimización de hiperparámetros: Modelo base Dummy, búsqueda de hiperparámetros con GridSearchCV (5-Fold CV) y evaluación comparativa.

* Interpretabilidad y negocio: Análisis SHAP y estimación del impacto financiero.

* Pasos omitidos:

    * Sobre o sub muestreo: Se omitió intencionalmente el sobre y sub muestreo de los datos, en su lugar se utilizó el balanceo por pesos de clase nativos de los algoritmos (scale_pos_weight / class_weight='balanced'), logrando los mejores resultados, además que utilizar métricas como ROC-AUC y F1 son adecuadas para evaluar modelos de clasificación binaria que tengan desbalance de clase. 

#### ¿Qué dificultades encontraste y cómo lograste resolverlas?

* Dificultad 1: Inconsistencia de formato en TotalCharges. Había valores en blanco (" ") y ceros almacenados como texto que impedían la conversión directa a flotante.

    * Solución: Se utilizó pd.to_numeric(..., errors='coerce') para convertir vacíos en NaN, sustituyéndolos luego de forma precisa con el valor correspondiente de MonthlyCharges del cliente.
      

* Dificultad 2: Riesgo de Data Leakage al escalar e imputar.

    * Solución: Se aseguró que el escalador (StandardScaler) realizara el fit exclusivamente sobre el subconjunto de entrenamiento (X_train), aplicando solo transform sobre prueba y validación.

#### ¿Cuáles fueron algunos de los pasos clave para resolver la tarea?

* La creación de TenureDays: Transformar las fechas de inicio y fin en la permanencia total en días fue la variable individual que más aportó al poder predictivo del modelo.

* El ajuste del peso de clases (scale_pos_weight): Permitió que el algoritmo penalizara los errores en la clase minoritaria (Churn), elevando el Recall del modelo del ~0% (Baseline) al 80.48%.

* Uso de la arquitectura LightGBM: La capacidad de los árboles basados en gradiente para capturar relaciones no lineales entre variables continuas y categóricas fue superior a los modelos lineales y a Random Forest.

#### ¿Cuál es tu modelo final y qué nivel de calidad tiene?

**Modelo Final**: LightGBM Classifier calibrado mediante GridSearchCV (n_estimators=100, learning_rate=0.1, num_leaves=31, scale_pos_weight=2.77).

* Nivel de Calidad (Evaluado en el conjunto de validación final no visto):

    * ROC-AUC: 0.9114 (Capacidad sobresaliente de discriminación entre clases).

    * Recall: 80.48% (Detecta a 4 de cada 5 clientes en riesgo real de irse).

    * Precision: 65.43% (Buen control de costos por falsos positivos).

    * F1-Score: 0.7218 / Accuracy: 83.53%.